# Silver layer - weather metrics

Aggregate FMI weather observations from the Bronze layer into 15-minute
station-level metrics. This step applies minimal cleaning and produces
windowed weather signals for downstream Gold KPI and pipeline health tables.


In [0]:
from pyspark.sql import functions as F

_ = spark.sql("USE azure_streaming_mvp")

### Processing parameters
The parameters below control the weather aggregation window
and the lookback horizon for recent FMI observations.

In [0]:
WEATHER_WINDOW = "15 minutes"
WEATHER_LOOKBACK_MINUTES = 360  # 6 hours

### Read and prepare recent weather events

Load weather events from the shared Bronze table, keep recent FMI observations,
and derive a station-level identifier for aggregation

In [0]:
bronze_all = spark.table("bronze_events")

weather_recent = (
    bronze_all
    .filter(F.col("source") == F.lit("fmi_weather"))
    .filter(
        F.col("event_time_ts") >= (
            F.current_timestamp()
            - F.expr(f"INTERVAL 1 MINUTE * {WEATHER_LOOKBACK_MINUTES}")
        )
    )
    .withColumn("station_id", F.col("entity_id"))   # entity_id already is fmisid or place
)

### Basic cleaning

Keep valid weather values and remove clearly unrealistic temperature outliers for the MVP.

In [0]:
# Basic filtering for MVP
weather_clean = (
    weather_recent
    .filter(F.col("value").isNotNull())   # keep valid observations
    # Filter extreme temperature values
    .filter(
        ~(
            (F.col("metric") == F.lit("t2m")) &
            ((F.col("value") < -60) | (F.col("value") > 60))
        )
    )
)

### Windowed aggregation

Aggregate cleaned weather observations into 15-minute station-level windows.

In [0]:
silver_weather = (
    weather_clean
    .groupBy(
        F.window("event_time_ts", WEATHER_WINDOW).alias("window"),
        F.col("metric"),
        F.col("station_id")
    )
    .agg(
        F.avg("value").alias("avg_value"),
        F.count(F.lit(1)).alias("n_events"),
        F.avg(
            F.greatest(
                F.unix_timestamp("ingest_time_ts") - F.unix_timestamp("event_time_ts"),
                F.lit(0)
            )
        ).alias("avg_ingest_delay_sec")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        F.col("metric"),
        F.col("station_id"),
        F.col("avg_value"),
        F.col("n_events"),
        F.col("avg_ingest_delay_sec")
    )
)

### Persist Silver output

Write the aggregated weather metrics to the Silver layer.

In [0]:
# Persist Silver output
(silver_weather.write.mode("overwrite").saveAsTable("silver_weather_metrics"))

display(spark.table("silver_weather_metrics")
        .orderBy(F.col("window_start").desc(), F.col("station_id"))
        .limit(20)
)

window_start,window_end,metric,station_id,avg_value,n_events,avg_ingest_delay_sec
2026-03-08T15:30:00.000Z,2026-03-08T15:45:00.000Z,t2m,100971,4.4,2,674.0
2026-03-08T15:15:00.000Z,2026-03-08T15:30:00.000Z,t2m,100971,4.4,1,1574.0
2026-03-08T15:00:00.000Z,2026-03-08T15:15:00.000Z,t2m,100971,4.800000000000001,2,2474.0
2026-03-08T14:45:00.000Z,2026-03-08T15:00:00.000Z,t2m,100971,5.1,1,3374.0
2026-03-08T14:30:00.000Z,2026-03-08T14:45:00.000Z,t2m,100971,5.55,2,4274.0
2026-03-08T14:15:00.000Z,2026-03-08T14:30:00.000Z,t2m,100971,5.7,1,5174.0
2026-03-08T14:00:00.000Z,2026-03-08T14:15:00.000Z,t2m,100971,5.6,2,6074.0
2026-03-08T13:45:00.000Z,2026-03-08T14:00:00.000Z,t2m,100971,6.1,1,6974.0
2026-03-08T13:30:00.000Z,2026-03-08T13:45:00.000Z,t2m,100971,6.35,2,7874.0
2026-03-08T13:15:00.000Z,2026-03-08T13:30:00.000Z,t2m,100971,6.7,1,8774.0


## Data Validation checks

These checks confirm that the Silver weather aggregation produced
stable windowed outputs without duplicate keys.

In [0]:
# Row count
spark.sql("""
SELECT COUNT(*) AS row_count
FROM silver_weather_metrics
""").show()

+---------+
|row_count|
+---------+
|       10|
+---------+



In [0]:
# Latest window
spark.sql("""
SELECT MAX(window_end) AS latest_window_end
FROM silver_weather_metrics
""").show()

+-------------------+
|  latest_window_end|
+-------------------+
|2026-03-08 15:45:00|
+-------------------+



In [0]:
# Check duplicate keys
spark.sql("""
SELECT window_start, metric, station_id, COUNT(*) AS row_count
FROM silver_weather_metrics
GROUP BY window_start, metric, station_id
HAVING row_count > 1
""").show(truncate=False)

+------------+------+----------+---------+
|window_start|metric|station_id|row_count|
+------------+------+----------+---------+
+------------+------+----------+---------+



In [0]:
# ---- Notebook completion signal ----
dbutils.notebook.exit("OK")